# 🏥 TrialMatch AI — Criteria Parser Fine-Tuning

**Task**: Classify eligibility criteria text into 6 categories (complexity × type)

**Method**: LoRA (PEFT) on BioBERT — trains only ~0.5% of parameters

**Runtime**: GPU required → Runtime → Change runtime type → **T4 GPU**

```
train.jsonl → Tokenize → BioBERT + LoRA → Train → Evaluate → Merge → Download
```

In [ ]:
# Step 1: Install
!pip install -q transformers datasets evaluate accelerate peft scikit-learn

In [ ]:
# Step 2: Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Upload train.jsonl
from google.colab import files
print("Upload train.jsonl from fine_tuning/data/criteria_parser/")
uploaded = files.upload()

In [ ]:
# Step 4: Config
CONFIG = {
    "base_model": "dmis-lab/biobert-v1.1",
    "max_seq_length": 256,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["query", "value"],
    "num_epochs": 15,
    "batch_size": 16,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "num_labels": 6,
    "label_names": ["simple_inclusion", "simple_exclusion", "compound_inclusion", "compound_exclusion", "conditional_inclusion", "conditional_exclusion"],
    "output_dir": "./criteria_parser_lora",
    "merged_dir": "./criteria_parser_merged",
}
print(f"✅ Model: {CONFIG['base_model']}, LoRA r={CONFIG['lora_r']}, Epochs: {CONFIG['num_epochs']}")

In [ ]:
# Step 5: Load and preprocess data
import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

examples = []
with open("train.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        complexity_map = {"simple": 0, "compound": 1, "conditional": 2}
        type_map = {"inclusion": 0, "exclusion": 1}
        output = item["output"]
        label = complexity_map.get(output.get("complexity", "simple"), 0) * 2 + type_map.get(output.get("criteria_type", "inclusion"), 0)
        examples.append({"text": item["input"], "label": label, "full_output": json.dumps(output)})

print(f"Total examples: {len(examples)}")

train_data, val_data = train_test_split(examples, test_size=0.2, random_state=42)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
for i, name in enumerate(CONFIG["label_names"]):
    count = sum(1 for e in examples if e["label"] == i)
    print(f"  {i}: {name} → {count}")

In [ ]:
# Step 6: Tokenize
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=CONFIG["max_seq_length"])

train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "full_output"])
val_tokenized = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "full_output"])
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")
print(f"✅ Tokenized")

In [ ]:
# Step 7: Load model (fp16, no quantization)
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=CONFIG["num_labels"],
    torch_dtype=torch.float16,
).to("cuda")

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {CONFIG['base_model']}")
print(f"   Parameters: {total_params:,}")

In [ ]:
# Step 8: Apply LoRA (PEFT)
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100 * trainable / total
print(f"✅ LoRA applied")
print(f"   Trainable: {trainable:,} ({pct:.2f}%)")
print(f"   Frozen: {total - trainable:,}")

In [ ]:
# Step 9: Training setup
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
            "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]}

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_tokenized, eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)
print("✅ Trainer ready")

In [ ]:
# Step 10: Train!
print("🚀 Training...\n")
train_result = trainer.train()
print(f"\n✅ Done! Loss: {train_result.training_loss:.4f}, Steps: {train_result.global_step}")

In [ ]:
# Step 11: Evaluate
eval_results = trainer.evaluate()
print("\n📊 EVALUATION RESULTS")
for k, v in eval_results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

In [ ]:
# Step 12: Merge LoRA into base model
from peft import PeftModel
import os

print("Merging LoRA adapter...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"], num_labels=CONFIG["num_labels"], torch_dtype=torch.float16,
)

checkpoints = [d for d in os.listdir(CONFIG["output_dir"]) if d.startswith("checkpoint-")]
best = os.path.join(CONFIG["output_dir"], sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]) if checkpoints else CONFIG["output_dir"]
print(f"Loading from: {best}")

merged = PeftModel.from_pretrained(base_model, best).merge_and_unload()
merged.save_pretrained(CONFIG["merged_dir"])
tokenizer.save_pretrained(CONFIG["merged_dir"])
print(f"✅ Saved to {CONFIG['merged_dir']}")

In [ ]:
# Step 13: Save metrics
metrics = {
    "model": CONFIG["base_model"], "method": "LoRA (PEFT)",
    "lora_rank": CONFIG["lora_r"], "lora_alpha": CONFIG["lora_alpha"],
    "trainable_pct": round(pct, 2), "epochs": CONFIG["num_epochs"],
    "train_loss": round(train_result.training_loss, 4),
    "eval": {k: round(v, 4) if isinstance(v, float) else v for k, v in eval_results.items()},
    "train_samples": len(train_dataset), "val_samples": len(val_dataset),
}
with open(f"{CONFIG['merged_dir']}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(f"{CONFIG['merged_dir']}/lora_config.json", "w") as f:
    json.dump({"r": CONFIG["lora_r"], "alpha": CONFIG["lora_alpha"], "method": "LoRA via PEFT"}, f, indent=2)
print("✅ Metrics saved")
print(json.dumps(metrics, indent=2))

In [ ]:
# Step 14: Test inference
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline("text-classification", model=CONFIG["merged_dir"], tokenizer=CONFIG["merged_dir"], device=0)

tests = [
    "Patients must be at least 18 years of age.",
    "No prior treatment with any anti-PD-1 antibody.",
    "ANC >= 1500/uL, platelets >= 100000/uL, hemoglobin >= 9.0 g/dL.",
    "Total bilirubin <= 1.5 x ULN OR direct bilirubin <= ULN if total > 1.5 x ULN.",
]
print("\n🧪 INFERENCE TEST")
for text in tests:
    r = classifier(text)
    idx = int(r[0]["label"].split("_")[-1])
    name = CONFIG["label_names"][idx] if idx < len(CONFIG["label_names"]) else r[0]["label"]
    print(f"  {text[:60]}...")
    print(f"    → {name} ({r[0]['score']:.3f})")

In [ ]:
# Step 15: Zip and download
import shutil
shutil.make_archive("criteria_parser_v1", "zip", ".", CONFIG["merged_dir"])

print("📦 Files:")
for f in sorted(os.listdir(CONFIG["merged_dir"])):
    print(f"   {f} ({os.path.getsize(os.path.join(CONFIG['merged_dir'], f))/1024:.1f} KB)")

print("\n🎯 Unzip into: trialmatch-ai/fine_tuning/models/criteria_parser/")
files.download("criteria_parser_v1.zip")